# Taking the waves dataset and turning it into a graph

original dataset info:
- dataset: https://www.kaggle.com/datasets/thedevastator/marine-institute-buoy-wave-forecast/data
- provenance: https://data.gov.ie/dataset/marine-institute-buoy-wave-forecast
- location: off the coast of Ireland
- buoys: 16 stations

In [121]:
import tsl
from tsl.data import Data

In [122]:
print(f"tsl version: {tsl.__version__}")

tsl version: 0.9.5


In [123]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datetime import datetime
from tsl.data import SpatioTemporalDataset
from tsl.datasets import Dataset

In [124]:
data = torch.load('../../experiments/data/pt/group/icbn.pt')
dataset = data['data']
aux = data['time']
target = data['target']

dataset.shape, target.shape

(torch.Size([385, 9, 12]), torch.Size([385, 9, 1]))

In [125]:
train_ratio = 0.8
train_size = int(train_ratio * dataset.shape[0])

train_data = dataset[:train_size]
test_data = dataset[train_size:]

train_target = target[:train_size]
test_target = target[train_size:]

train_data.shape, test_data.shape, train_target.shape, test_target.shape

(torch.Size([308, 9, 12]),
 torch.Size([77, 9, 12]),
 torch.Size([308, 9, 1]),
 torch.Size([77, 9, 1]))

In [126]:
from torch_geometric.utils import dense_to_sparse

def create_fully_connected_edge_index(num_nodes):
    # Create a dense adjacency matrix for a fully connected graph
    adj = torch.ones((num_nodes, num_nodes), dtype=torch.float)
    
    # Convert the dense adjacency matrix to a sparse edge_index
    edge_index, _ = dense_to_sparse(adj)
    
    return edge_index

In [127]:
tsl_train = SpatioTemporalDataset(target=train_target, horizon=1, window=6, stride=1)
tsl_train.edge_index = create_fully_connected_edge_index(9)

tsl_test = SpatioTemporalDataset(target=test_target, horizon=1, window=6, stride=1)
tsl_test.edge_index = create_fully_connected_edge_index(9)

tsl_train[0], tsl_test[0]

(Data(
   input=(x=[t=6, n=9, f=1], edge_index=[2, e=81]),
   target=(y=[t=1, n=9, f=1]),
   has_mask=False
 ),
 Data(
   input=(x=[t=6, n=9, f=1], edge_index=[2, e=81]),
   target=(y=[t=1, n=9, f=1]),
   has_mask=False
 ))

In [128]:
tsl_train.add_covariate('multivariate', train_data)
tsl_test.add_covariate('multivariate', test_data)

In [129]:
tsl_train[0], tsl_test[0]

(Data(
   input=(x=[t=6, n=9, f=1], multivariate=[t=6, n=9, f=12], edge_index=[2, e=81]),
   target=(y=[t=1, n=9, f=1]),
   has_mask=False
 ),
 Data(
   input=(x=[t=6, n=9, f=1], multivariate=[t=6, n=9, f=12], edge_index=[2, e=81]),
   target=(y=[t=1, n=9, f=1]),
   has_mask=False
 ))

### Time then Space paradigm

In [130]:
import torch.nn as nn

from tsl.nn.blocks.encoders import RNN
from tsl.nn.layers import NodeEmbedding, GraphConv
from einops.layers.torch import Rearrange

In [131]:
class TimeThenSpaceModel(nn.Module):
    def __init__(self, num_nodes, num_features, hidden_size, horizon, num_layers):
        super(TimeThenSpaceModel, self).__init__()
        
        self.encoder = nn.Linear(num_features, hidden_size)
        self.node_embedding = NodeEmbedding(n_nodes=num_nodes, emb_size=hidden_size)
        
        self.time_nn = RNN(input_size=hidden_size, hidden_size=hidden_size, num_layers=num_layers)
        self.space_nn = GraphConv(input_size=hidden_size, output_size=hidden_size, activation='relu')
        
        self.decoder = nn.Linear(hidden_size, 1 * horizon)
        
        self.rearrange = Rearrange('b n (t f) -> b t n f', t=horizon)
        # self.rearrange = Rearrange('b t n f -> b n (t f)', t=horizon)
        
    def forward(self, x, edge_index):
        x_enc = self.encoder(x)
        x_emb = x_enc + self.node_embedding()
        x_emb = x_emb.unsqueeze(0)
        ht = self.time_nn(x_emb)
        # print(ht.shape)
        hs = self.space_nn(ht, edge_index)
        z = self.decoder(hs)
        
        # z = z.squeeze(0)
        # z = self.rearrange(z)
        
        return z

In [132]:
hidden_size = 4
time_layers = 1

num_features = tsl_train.n_channels
num_nodes = tsl_train.n_nodes
horizon = tsl_train.horizon

stgnn = TimeThenSpaceModel(
    num_nodes=num_nodes,
    num_features=num_features,
    hidden_size=hidden_size,
    horizon=horizon,
    num_layers=time_layers
)

print(stgnn)

TimeThenSpaceModel(
  (encoder): Linear(in_features=1, out_features=4, bias=True)
  (node_embedding): NodeEmbedding(n_nodes=9, embedding_size=4)
  (time_nn): RNN(
    (rnn): GRU(4, 4)
  )
  (space_nn): GraphConv(4, 4)
  (decoder): Linear(in_features=4, out_features=1, bias=True)
  (rearrange): Rearrange('b n (t f) -> b t n f', t=1)
)


In [133]:
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(stgnn.parameters(), lr=0.01)

stgnn.train()

for epoch in range(20):
    stgnn.train()
    epoch_loss = 0
    for i, data in enumerate(tsl_train):
        optimizer.zero_grad()
        x = data.x
        edge_index = data.edge_index
        y = data.y
        
        y_hat = stgnn(x, edge_index)
        
        loss = criterion(y_hat, y)
        epoch_loss += loss.item()
        
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1} - Loss: {epoch_loss/len(tsl_train):.2f}")
    

Epoch 1 - Loss: 0.48
Epoch 2 - Loss: 0.12
Epoch 3 - Loss: 0.13
Epoch 4 - Loss: 0.12
Epoch 5 - Loss: 0.11
Epoch 6 - Loss: 0.11
Epoch 7 - Loss: 0.10
Epoch 8 - Loss: 0.11
Epoch 9 - Loss: 0.11
Epoch 10 - Loss: 0.10
Epoch 11 - Loss: 0.10
Epoch 12 - Loss: 0.10
Epoch 13 - Loss: 0.10
Epoch 14 - Loss: 0.10
Epoch 15 - Loss: 0.11
Epoch 16 - Loss: 0.10
Epoch 17 - Loss: 0.12
Epoch 18 - Loss: 0.12
Epoch 19 - Loss: 0.12
Epoch 20 - Loss: 0.12
